# Case Study II: 2024 KATL Taxiway Collision:

 - The incident occurred on Sept. 10 at approximately 10:07 a.m. as two aircraft, Delta Air Lines flight 295, an Airbus A350-941 (N503DN), and Endeavor Air flight 5526 (operating as Delta), a Mitsubishi CRJ-900 (N302PQ), collided while taxiing for departure.
 - https://www.fox5atlanta.com/news/ntsb-preliminary-report-two-delta-planes-collided-atlanta-airports-taxiway

In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)
import spacy

nlp_ner = spacy.load("./transformer/model-best")
ruler = nlp_ner.add_pipe(
    "entity_ruler",
    after="ner",  # or before="ner"
    config={"overwrite_ents": True}  
)
ruler.from_disk("entity_rulers.jsonl")

text = "295 heavy turn on foxtrot hold short of ramp 5."
doc = nlp_ner(text)

print("Entities found:")
for ent in doc.ents:
    print(f" - {ent.text} : {ent.label_}")

Entities found:
 - 295 : CALLSIGN
 - turn : ACSTATE
 - foxtrot : DESTINATION
 - hold : ACSTATE
 - ramp 5 : DESTINATION


In [2]:
import re
import glob
import pandas as pd
import spacy
import torch
import numpy as np
from sentence_transformers import SentenceTransformer, util

###############################################################################
# 1. Runway Pattern Functions
###############################################################################
# This regex finds an optional "runway" followed by 1–2 digits and an optional L/R.
# It requires a whitespace, end-of-string, or common punctuation after the designator.

# def extract_all_runway_designators(text: str) -> list:
#     """
#     Return all runway designators found in the text as a list.
#     The designator must be one or two digits followed immediately by either L or R.
    
#     For example:
#       "17:43:12 ... runway 34R, continue approach" 
#     will return: ["34R"]
    
#     All designators are returned in uppercase.
#     """
#     matches = RUNWAY_REGEX.findall(text)
#     return [m.upper() for m in matches if m.strip()]

RUNWAY_REGEX = re.compile(
    r'(?i)(?:runway\s+)?(?P<number>\d{1,2})\s*(?:(?P<side>[LR])|(?P<sideWord>right|left))(?=\s|$|[.,;:!])'
)

def extract_all_runway_designators(text: str) -> list:
    """
    Return all runway designators found in the text as a list.
    This version recognizes designators written either as "34R" or as "8 right" (which is normalized to "08R").
    """
    designators = []
    for match in RUNWAY_REGEX.finditer(text):
        number = match.group("number")
        side = match.group("side")
        side_word = match.group("sideWord")
        # Format the number to always be two digits (e.g., "8" -> "08")
        number = number.zfill(2)
        if side_word:
            side = side_word[0].upper()  # Convert "right" or "left" to "R" or "L"
        if side:
            designators.append(f"{number}{side}")
        else:
            designators.append(number)
    return designators

def is_runway_pattern(text: str) -> bool:
    """Return True if at least one runway designator is found in the text."""
    return len(extract_all_runway_designators(text)) > 0

def format_runway_code(designator: str) -> str:
    """
    Format the runway designator by prepending 'RW'.
    For example, "8R" becomes "RW8R".
    """
    return f"RW{designator}"

def find_runway_entry_node(df: pd.DataFrame, runway_designator: str) -> str | None:
    """
    Given a runway designator (e.g. "8R"), look up the CSV row where that runway
    appears as an entry. In the CSV we assume that the runway entry is indicated by:
      - Either: refName1 equals the formatted runway code and type1 equals "Entry"
      - Or: refName2 equals the formatted runway code and type2 equals "Entry"
    Returns the node 'id' of the first matching row, or None if not found.
    """
    runway_code = format_runway_code(runway_designator)  # e.g. "RW8R"
    mask_ref1 = (df['refName1'] == runway_code) & (df['type1'].str.lower() == 'entry')
    mask_ref2 = (df['refName2'] == runway_code) & (df['type2'].str.lower() == 'entry')
    match_mask = mask_ref1 | mask_ref2
    matched_rows = df[match_mask]
    if not matched_rows.empty:
        return matched_rows.iloc[0]['id']
    return None

###############################################################################
# 2. Node Similarity Functions for Non-Runway Destinations
###############################################################################
def build_id_embeddings(df: pd.DataFrame, model_name='sentence-transformers/all-MiniLM-L6-v2'):
    """
    Build embeddings for the 'id' column of the CSV.
    Returns a tuple (model, embeddings).
    """
    model = SentenceTransformer(model_name)
    texts = df['id'].astype(str).tolist()
    embeddings = model.encode(texts, convert_to_tensor=True)
    return model, embeddings

def find_topk_similar_nodes(query: str, df: pd.DataFrame, model, embeddings, top_k=5):
    """
    Compute top-k similarity between the query and the embedded node IDs.
    Returns a list of dictionaries with the node id and similarity score.
    """
    query_emb = model.encode(query, convert_to_tensor=True)
    cos_scores = util.cos_sim(query_emb, embeddings)[0]
    top_results = torch.topk(cos_scores, k=top_k)
    results = []
    all_ids = df['id'].tolist()
    for i, idx_tensor in enumerate(top_results.indices):
        idx = idx_tensor.item()
        score = top_results.values[i].item()
        results.append({
            "id": all_ids[idx],
            "similarity_score": float(score)
        })
    return results

###############################################################################
# 3. Main Function: Build the Meta Table from the Transcript
###############################################################################
if __name__ == "__main__":
    # --- Load your pretrained spaCy model and add your entity ruler ---
    nlp_ner = spacy.load("./transformer/model-best")
    # Add the entity ruler (adjust the pipe position as needed)
    ruler = nlp_ner.add_pipe("entity_ruler", after="ner", config={"overwrite_ents": True})
    ruler.from_disk("entity_rulers.jsonl")
    
    # --- Load the CSV with airport nodes ---
    ICAO = 'KATL'
    airport_nodes = f'./Airport Layouts/{ICAO}_Nodes_Def.csv'
    df = pd.read_csv(airport_nodes)
    
    # Build embeddings for non-runway similarity search on the node "id" column.
    model_id, id_embeddings = build_id_embeddings(df)
    
    # List to collect meta table rows.
    meta_rows = []
    
    # --- Process transcript files ---
    test_file_paths = glob.glob('/home/yp6443/research/nlp/voice_data/test_file/*.txt')
    file_idx = 0  # adjust as needed

    with open(test_file_paths[file_idx], 'r') as file:
        for line in file:
            line_text = line.strip()
            if not line_text:
                continue
            
            # Extract the time as the first token in the line.
            time_val = line_text.split()[0]
            
            # Run your spaCy NER on the line.
            doc = nlp_ner(line_text)
            print(f"\nLine: \"{line_text}\"")
            print("Entities found:")
            for ent in doc.ents:
                print(f"  - {ent.text} : {ent.label_}")
            
            # Initialize fields (if an entity is missing, the field remains empty).
            callsign = ""
            destination_text = ""
            ac_state_list = []
            
            # Extract entity values (adjust label names as used by your model).
            for ent in doc.ents:
                if ent.label_ == "CALLSIGN":
                    callsign = ent.text
                elif ent.label_ == "DESTINATION":
                    destination_text = ent.text
                elif ent.label_ == "ACSTATE":  # or "AC_STATE" if that is your model's label
                    ac_state_list.append(ent.text)
            ac_state = ",".join(ac_state_list)
            
            # --- Determine the destination runway and final destination ---
            # First, try applying the regex to the entire line.
            designators_line = extract_all_runway_designators(line_text)
            if designators_line:
                # If found in the entire line, choose the last designator.
                chosen = designators_line[-1]
                dest_runway = chosen
                entry_node = find_runway_entry_node(df, chosen)
                if entry_node:
                    final_destination = entry_node
                else:
                    final_destination = f"No entry found for runway {chosen}"
            else:
                # Otherwise, fall back to checking the DESTINATION entity.
                if destination_text and is_runway_pattern(destination_text):
                    designators = extract_all_runway_designators(destination_text)
                    chosen = designators[-1] if designators else None
                    if chosen:
                        dest_runway = chosen
                        entry_node = find_runway_entry_node(df, chosen)
                        if entry_node:
                            final_destination = entry_node
                        else:
                            final_destination = f"No entry found for runway {chosen}"
                    else:
                        final_destination = destination_text
                        dest_runway = ""
                else:
                    dest_runway = ""
                    # For non-runway orders, use the destination_text.
                    final_destination = destination_text
            
            # Build a meta table row.
            meta_row = {
                "callsign": callsign,
                "time": time_val,
                "ac_state": ac_state,
                "dest_runway": dest_runway,
                "destination": final_destination
            }
            meta_rows.append(meta_row)
    
    # Create and print the meta table.
    meta_df = pd.DataFrame(meta_rows, columns=["callsign", "time", "ac_state", "dest_runway", "destination"])
    
    # -----------------------
    # Post-Processing Step:
    # For each callsign, if a dest_runway is provided on any row, propagate it to all rows of that callsign.
    # Replace empty strings with NaN for proper filling.
    meta_df['dest_runway'] = meta_df['dest_runway'].replace('', np.nan)
    # For each callsign, forward-fill then backward-fill the dest_runway.
    meta_df['dest_runway'] = meta_df.groupby('callsign')['dest_runway'].transform(lambda x: x.ffill().bfill())
    # Replace NaN back with an empty string if desired.
    meta_df['dest_runway'] = meta_df['dest_runway'].fillna('')
    # -----------------------
    meta_df = meta_df[~meta_df['time'].isin(['N/A'])]
    
    print("\nFinal Meta Table:")
    print(meta_df.reset_index(drop=True))



Line: "0:08 And Delta 295 heavy taxi with Romeo."
Entities found:
  - Delta 295 : CALLSIGN
  - taxi : ACSTATE
  - Romeo : DESTINATION

Line: "0:14 Delta 295 heavy Atlanta ground runway 8 right taxi golf short of foxtrot."
Entities found:
  - Delta 295 : CALLSIGN
  - runway 8 right : CALLSIGN
  - taxi : ACSTATE
  - foxtrot : DESTINATION

Line: "0:20 Taxi via golf short of foxtrot delta 295."
Entities found:
  - Taxi : ACSTATE
  - foxtrot : DESTINATION
  - delta 295 : CALLSIGN

Line: "0:33 Delta 295 heavy continue via a left turn on foxtrot hold short of ramp 5."
Entities found:
  - Delta 295 : CALLSIGN
  - continue : ACSTATE
  - foxtrot : DESTINATION
  - hold : ACSTATE
  - ramp 5 : DESTINATION

Line: "0:36 Alright, make a left on Foxtrot short of ramp 5 Delta 295 thank you."
Entities found:
  - Foxtrot : DESTINATION
  - ramp 5 : DESTINATION
  - Delta 295 : CALLSIGN

Line: "0:44 Delta 295 heavy ramp 5 give way to that opposite direction 717 inbound and then join Echo."
Entities found:
 

In [3]:
print("Meta Table:")
pd.set_option('display.max_rows', None)
meta_df

Meta Table:


,callsign,time,ac_state,dest_runway,destination
0,Delta 295,0:08,taxi,,Romeo
1,runway 8 right,0:14,taxi,08R,Rwy_02_001
2,delta 295,0:20,Taxi,,foxtrot
3,Delta 295,0:33,"continue,hold",,ramp 5
4,Delta 295,0:36,,,ramp 5
5,Delta 295,0:44,"give way,inbound,join",,Echo
6,Delta 295,0:50,give way,,
7,Endeavor 5526,0:57,taxi,08R,Rwy_02_001
8,endeavor 5526,1:01,,,
9,Delta 295,1:03,,,


In [ ]:
accident_time = "17:47:30"
accident_callsign = "Japan Air 516"
new_row_1 = {
        "callsign": accident_callsign,
        "time": accident_time,
        "ac_state": "collision",
        "dest_runway": "",      # Could be left empty if not available.
        "destination": ""
        }

meta_df = pd.concat([meta_df, pd.DataFrame([new_row_1])], ignore_index=True)
    
accident_callsign = "JA722A"
new_row_2 = {
        "callsign": accident_callsign,
        "time": accident_time,
        "ac_state": "collision",
        "dest_runway": "",      # Could be left empty if not available.
        "destination": ""
        }
meta_df = pd.concat([meta_df, pd.DataFrame([new_row_2])], ignore_index=True)

meta_df